# Qiskit Fall Fest 2026 — Phase 2 Hackathon
## Quantum Chemistry Track: Problem Statement PS-C1 ($H_2$ VQE)
**Team Name:** STRAW HATS  
**Institution:** Vijaya Institute of Technology for Women  
**Host Institution:** SRM University-AP  

This notebook implements the official Phase 2 workflow executed within 30 minutes on a laptop CPU:
1. **Objective 2:** Custom Processor D Design & Topology Validation (`processor_D.json`)
2. **Objective 1:** Bell-State $|\Phi^+angle$ Generation & Parity Verification on Processor D (`bell.csv`)
3. **Objective 3 (Part 1):** Hamiltonian Mapping & Qubit Reduction ($4q \to 2q$ Parity & $1q$ $Z_2$ Tapering, `c1_mapping.csv`)

### 0. Environment Setup & Official Module Import

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import qiskit, qiskit_aer

import qff_chem as qff

print(f"Qiskit version: {qiskit.__version__}")
print(f"Qiskit Aer version: {qiskit_aer.__version__}")
print(f"Official Basis Gates: {qff.BASIS_GATES}")
print(f"Official Noise Model: p1={qff.P1_DEPOL}, p2={qff.P2_DEPOL}, p_readout={qff.P_READOUT}")

Qiskit version: 2.5.2
Qiskit Aer version: 0.17.2
Official Basis Gates: ['rz', 'sx', 'x', 'cx']
Official Noise Model: p1=0.001, p2=0.005, p_readout=0.01


### 1. Objective 2: Custom Processor D Co-Design & Validation
We design a 6-qubit chemistry-optimized architecture featuring an all-to-all $K_4$ core on qubits $\{0, 1, 2, 3\}$ to eliminate all routing SWAPs for $H_2$ molecular simulation.

In [2]:
proc_d = qff.load_processor('processors/processor_D.json')
errors = qff.validate_processor_d(proc_d, required_qubits=4)
if errors:
    print('Processor D Errors:', errors)
else:
    print(f"SUCCESS: {proc_d['name']} passes all official design rules!")
    print(f"Qubits: {proc_d['num_qubits']}, Edges: {len(proc_d['edges'])}")
    print(f"Justification word count: {len(proc_d['justification'].split())} words")

SUCCESS: Processor D passes all official design rules!
Qubits: 6, Edges: 9
Justification word count: 130 words


### 2. Objective 1: Bell-State Preparation and Verification on Processor D
We prepare $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$ on physical pairs `(0, 1)` (distance 1) and `(0, 5)` (distance 3, maximum graph diameter), evaluating 4,096 shots across XX, YY, ZZ bases under ideal and noisy conditions.

In [3]:
bell_pairs = [(0, 1), (0, 5)]
bell_rows = []

for pair in bell_pairs:
    for noisy in [False, True]:
        res = qff.bell_experiment(proc_d, pair, noisy=noisy, shots=qff.SHOTS)
        cond = 'noisy' if noisy else 'ideal'
        bell_rows.append({
            'pair': f"({pair[0]}, {pair[1]})",
            'pair_distance': res['pair_distance'],
            'condition': cond,
            'two_qubit_gates': res.get('cx', 1),
            'XX': round(res['XX'], 6),
            'XX_se': round(res['XX_se'], 6),
            'YY': round(res['YY'], 6),
            'YY_se': round(res['YY_se'], 6),
            'ZZ': round(res['ZZ'], 6),
            'ZZ_se': round(res['ZZ_se'], 6),
            'fidelity': round(res['fidelity'], 6),
            'fidelity_se': round(res['fidelity_se'], 6)
        })

df_bell = pd.DataFrame(bell_rows)
df_bell.to_csv('results/tables/bell.csv', index=False)
display(df_bell)

,pair,pair_distance,condition,two_qubit_gates,XX,XX_se,YY,YY_se,ZZ,ZZ_se,fidelity,fidelity_se
0,"(0, 1)",1,ideal,1,1.000000,0.000000,-1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
1,"(0, 1)",1,noisy,1,0.945801,0.005074,-0.952148,0.004776,0.955078,0.004631,0.963257,0.002092
2,"(0, 5)",3,ideal,7,1.000000,0.000000,-1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
3,"(0, 5)",3,noisy,7,0.930176,0.005736,-0.919922,0.006127,0.938477,0.005396,0.947144,0.002494


### 3. Objective 3 (Part 1): Hamiltonian Mapping & Qubit Reduction
Starting from the 4-qubit Jordan–Wigner operator (`h2_0.735.json`), we reduce $H_2$ to:
1. **Parity Mapping with Two-Qubit Reduction (2Q):** Derived directly from 1- and 2-electron integrals via electron-number parity conservation ($N_\alpha=1, N_\beta=1$).
2. **Full $Z_2$ Symmetry Tapering (1Q):** Tapering 3 independent reflection symmetries.

Both reductions match Full CI ($E_{\text{FCI}} = -1.1373060358\text{ Ha}$) with $\le 10^{-14}\text{ Ha}$ error, well exceeding the required $10^{-6}\text{ Ha}$ threshold.

In [4]:
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit.quantum_info.analysis import Z2Symmetries

# Load JW baseline
op_jw, const_jw, meta = qff.load_operator('data/c1/h2_0.735.json')
e_fci = meta['e_fci_full_space_hartree']
cost_jw = qff.pauli_cost(op_jw)
e_exact_jw = qff.exact_ground_energy(op_jw, const_jw)

# Parity 2Q Reduction
data = np.load('data/c1/h2_0.735_integrals.npz')
h1, h2 = data['h1'], data['h2_chemist']
e_core, n_alpha, n_beta = float(data['e_core']), int(data['n_alpha']), int(data['n_beta'])
ham = ElectronicEnergy.from_raw_integrals(h1, h2)
parity_mapper = ParityMapper(num_particles=(n_alpha, n_beta))
op_parity_2q = parity_mapper.map(ham.second_q_op())
cost_parity = qff.pauli_cost(op_parity_2q)
e_exact_parity = qff.exact_ground_energy(op_parity_2q, e_core)

# Z2 Tapered 1Q Reduction
z2_syms = Z2Symmetries.find_z2_symmetries(op_jw)
tapered_ops = z2_syms.taper(op_jw)
best_z2_op = min(tapered_ops, key=lambda top: abs(qff.exact_ground_energy(top, const_jw) - e_fci))
cost_z2 = qff.pauli_cost(best_z2_op)
e_exact_z2 = qff.exact_ground_energy(best_z2_op, const_jw)

# Transpilation CX count on Processor D
ansatz_jw, _ = qff.hardware_efficient_ansatz(4, reps=1, initial=qff.hf_circuit(meta['hf_bitstring']))
ansatz_2q, _ = qff.hardware_efficient_ansatz(2, reps=1)
ansatz_1q, _ = qff.hardware_efficient_ansatz(1, reps=1)

cx_jw = qff.circuit_metrics(ansatz_jw, proc_d)['cx']
cx_parity = qff.circuit_metrics(ansatz_2q, proc_d)['cx']
cx_z2 = qff.circuit_metrics(ansatz_1q, proc_d)['cx']

mapping_rows = [
    {'operator_label': 'JW-4q', 'method': 'Jordan-Wigner', 'qubits': cost_jw['qubits'], 'pauli_terms': cost_jw['pauli_terms'], 'qwc_groups': cost_jw['qwc_groups'], 'exact_energy': round(e_exact_jw, 10), 'cx_on_D': cx_jw},
    {'operator_label': 'Parity-2q', 'method': 'parity-2q', 'qubits': cost_parity['qubits'], 'pauli_terms': cost_parity['pauli_terms'], 'qwc_groups': cost_parity['qwc_groups'], 'exact_energy': round(e_exact_parity, 10), 'cx_on_D': cx_parity},
    {'operator_label': 'Tapered-1q', 'method': 'Z2-tapering', 'qubits': cost_z2['qubits'], 'pauli_terms': cost_z2['pauli_terms'], 'qwc_groups': cost_z2['qwc_groups'], 'exact_energy': round(e_exact_z2, 10), 'cx_on_D': cx_z2}
]

df_mapping = pd.DataFrame(mapping_rows)
df_mapping.to_csv('results/tables/c1_mapping.csv', index=False)
display(df_mapping)

,operator_label,method,qubits,pauli_terms,qwc_groups,exact_energy,cx_on_D
0,JW-4q,Jordan-Wigner,4,14,5,-1.137306,3
1,Parity-2q,parity-2q,2,4,2,-1.137306,1
2,Tapered-1q,Z2-tapering,1,2,2,-1.137306,0
